In [246]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from catboost import CatBoostClassifier
import joblib

In [183]:
df=pd.read_csv("bank-full.csv",delimiter=";")
df_main=df.copy()
df_label=df['y']
df.drop(['y'],axis=1,inplace=True)

#### no need to run this code the file is also present in the folder

In [184]:
# from ydata_profiling import ProfileReport
# profile = ProfileReport(df, title="Pandas Profiling Report", explorative=True)
# profile.to_file("report.html")

In [185]:
xtrain,xtest,ytrain,ytest=train_test_split(df,df_label,test_size=0.2,random_state=21)

In [186]:
num_df = xtrain.select_dtypes(include='number').copy()
num_coloumns=num_df.columns
num_df

,age,balance,day,duration,campaign,pdays,previous
1533,50,0,8,393,2,-1,0
39881,50,3370,2,160,2,-1,0
7432,36,0,29,211,6,-1,0
30441,55,1765,5,445,1,267,1
39017,28,352,18,431,1,-1,0
...,...,...,...,...,...,...,...
16432,57,215,23,288,9,-1,0
8964,56,62,5,243,1,-1,0
5944,30,306,26,372,14,-1,0
5327,28,370,23,108,5,-1,0


In [187]:
ordinal_cols = ['education', 'month']
nominal_cols = ['job','marital','default','housing','loan','contact','poutcome']

In [188]:
ordinal_df=xtrain[ordinal_cols].copy()
nominal_df=xtrain[nominal_cols].copy()

In [189]:
nominal_df

,job,marital,default,housing,loan,contact,poutcome
1533,technician,single,no,yes,no,unknown,unknown
39881,technician,married,no,no,no,cellular,unknown
7432,management,single,no,yes,no,unknown,unknown
30441,blue-collar,married,no,yes,no,cellular,failure
39017,admin.,single,no,yes,yes,cellular,unknown
...,...,...,...,...,...,...,...
16432,management,married,no,no,yes,cellular,unknown
8964,technician,married,no,no,yes,unknown,unknown
5944,unemployed,single,no,no,no,unknown,unknown
5327,technician,single,no,yes,no,unknown,unknown


In [190]:
num_pipeline =Pipeline([
    ("stdscaller",StandardScaler()),
    ])

In [191]:
ordinal_pipeline=Pipeline([
    ("ordinalencoder",OrdinalEncoder(categories=[
        ['unknown', 'primary', 'secondary', 'tertiary'],
        ['jan','feb','mar','apr','may','jun',
         'jul','aug','sep','oct','nov','dec']
    ]))
    ])

In [192]:
nominal_pipeline=Pipeline([
    ("onehotencoder",OneHotEncoder())
    ])

In [193]:
transformer=ColumnTransformer([
    ("num",num_pipeline,num_df.columns),
    ("ordinal",ordinal_pipeline,ordinal_df.columns),
    ("nominal",nominal_pipeline,nominal_df.columns),
],remainder='drop')

In [194]:
base_model=LogisticRegression()
lb=LabelEncoder()
train_data_y=lb.fit_transform(ytrain)
train_data=transformer.fit_transform(xtrain)

In [195]:
base_model.fit(train_data,train_data_y,)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [196]:
prediction_base=base_model.predict(transformer.fit_transform(xtest))

In [197]:
print(classification_report(lb.fit_transform(ytest),prediction_base))

              precision    recall  f1-score   support

           0       0.92      0.97      0.94      7983
           1       0.64      0.33      0.44      1060

    accuracy                           0.90      9043
   macro avg       0.78      0.65      0.69      9043
weighted avg       0.88      0.90      0.89      9043



In [203]:
category_feature = xtrain.select_dtypes(include=['object', 'category']).columns.tolist()

In [209]:
catboost = CatBoostClassifier(
    iterations=2000,
    depth=8,
    learning_rate=0.03,
    eval_metric='F1',
    auto_class_weights='Balanced'
)

In [210]:
catboost.fit(xtrain,ytrain,cat_features=category_feature)

0:	learn: 0.8133972	total: 74.5ms	remaining: 2m 28s
1:	learn: 0.8185015	total: 128ms	remaining: 2m 8s
2:	learn: 0.8252923	total: 194ms	remaining: 2m 9s
3:	learn: 0.8281179	total: 264ms	remaining: 2m 11s
4:	learn: 0.8322007	total: 342ms	remaining: 2m 16s
5:	learn: 0.8373545	total: 417ms	remaining: 2m 18s
6:	learn: 0.8395850	total: 488ms	remaining: 2m 18s
7:	learn: 0.8390607	total: 557ms	remaining: 2m 18s
8:	learn: 0.8366028	total: 630ms	remaining: 2m 19s
9:	learn: 0.8361258	total: 684ms	remaining: 2m 16s
10:	learn: 0.8365704	total: 743ms	remaining: 2m 14s
11:	learn: 0.8379817	total: 829ms	remaining: 2m 17s
12:	learn: 0.8341270	total: 889ms	remaining: 2m 15s
13:	learn: 0.8384227	total: 979ms	remaining: 2m 18s
14:	learn: 0.8380444	total: 1.06s	remaining: 2m 20s
15:	learn: 0.8382053	total: 1.14s	remaining: 2m 21s
16:	learn: 0.8402022	total: 1.23s	remaining: 2m 22s
17:	learn: 0.8418092	total: 1.31s	remaining: 2m 24s
18:	learn: 0.8415131	total: 1.39s	remaining: 2m 25s
19:	learn: 0.8423396	to

CatBoostClassifier(auto_class_weights='Balanced', depth=8, eval_metric='F1', iterations=2000, learning_rate=0.03)

In [211]:
prediction_catboost=catboost.predict(xtest)

In [212]:
print(classification_report(ytest,prediction_catboost))

              precision    recall  f1-score   support

          no       0.97      0.89      0.93      7983
         yes       0.50      0.81      0.62      1060

    accuracy                           0.88      9043
   macro avg       0.73      0.85      0.77      9043
weighted avg       0.92      0.88      0.89      9043



In [218]:
ytrain

1533      no
39881    yes
7432      no
30441     no
39017    yes
        ... 
16432     no
8964      no
5944      no
5327      no
15305     no
Name: y, Length: 36168, dtype: object

In [219]:
testcase=xtrain.head().copy()

In [229]:
testcase

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
1533,50,technician,single,tertiary,no,0,yes,no,unknown,8,may,393,2,-1,0,unknown
39881,50,technician,married,secondary,no,3370,no,no,cellular,2,jun,160,2,-1,0,unknown
7432,36,management,single,tertiary,no,0,yes,no,unknown,29,may,211,6,-1,0,unknown
30441,55,blue-collar,married,secondary,no,1765,yes,no,cellular,5,feb,445,1,267,1,failure
39017,28,admin.,single,secondary,no,352,yes,yes,cellular,18,may,431,1,-1,0,unknown


In [223]:
index=testcase.index

In [245]:
for i in (index):
    print('-----------------------------------------------------------------------------------------------------------')
    row=testcase.loc[i]
    print("Customer Case Study # :- ")
    print(row)
    print("The predicted value :- ",catboost.predict(row))
    print("The acutal value:- ",ytrain.loc[i])
    print()
    if catboost.predict(row)==ytrain.loc[i]:
        print('The prediction is correct !!!')
    else:
        print('The prediction is INCORRECT ???')
    print("")
    print()
    print("Probability of NO :- ",catboost.predict_proba(row)[0].round(4))
    print("Probability of YES:- ",catboost.predict_proba(row)[1].round(4))
    print("                                              ===================                         ")

-----------------------------------------------------------------------------------------------------------
Customer Case Study # :- 
age                  50
job          technician
marital          single
education      tertiary
default              no
balance               0
housing             yes
loan                 no
contact         unknown
day                   8
month               may
duration            393
campaign              2
pdays                -1
previous              0
poutcome        unknown
Name: 1533, dtype: object
The predicted value :-  no
The acutal value:-  no

The prediction is correct !!!


Probability of NO :-  0.9872
Probability of YES:-  0.0128
-----------------------------------------------------------------------------------------------------------
Customer Case Study # :- 
age                  50
job          technician
marital         married
education     secondary
default              no
balance            3370
housing              no
loan         

In [247]:
joblib.dump(catboost,'model.pkl')

['model.pkl']